# OpenAI Chat Completions API

> Translate Chat Completions requests, responses, and streams to and from fastllm's shared message types

In [ ]:
#| default_exp openai_chat

## Imports

In [ ]:
#| export
import json
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fasttransport.errors import api_error_from_event

from fastllm.types import *
from aidialog.msg_parts import (Part, PartType, Msg, Text, Thinking, Refusal, ToolUse, ToolResult, tool_text,
    InputImage, InputAudio, InputVideo, InputFile, sys_text, part_txt, data_url)
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream
from fastllm.openai_responses import norm_usage

In [ ]:
#| hide
import base64,httpx2
from fasttransport.core import AsyncHttpCli
from fastcore.test import *
from fastcore.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy, doms
from aidialog.msg_parts import fmt2hist, hist2fmt


In [ ]:
enable_cachy(doms=doms+sample_doms, hdrs=('content-type',))


The examples use `chat_call` to POST to `/chat/completions` through `provider_req`. Pass a different `AsyncHttpCli` to use another provider's compatible endpoint. The call returns a response object or, with `stream=True`, an event iterator.

Most examples call DeepSeek V4.1 Flash on Fireworks through `fw_cli`. OpenAI's GPT-6 models accept function tools on Chat Completions only with `reasoning_effort='none'`. OpenAI recommends the Responses API for these models. The audio and PDF examples show OpenAI-specific input formats. They call OpenAI through `oai_cli`.

In [ ]:
oai_cli = AsyncHttpCli('https://api.openai.com/v1', base_headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
fw_cli = AsyncHttpCli('https://api.fireworks.ai/inference/v1', base_headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
ds = 'accounts/fireworks/models/deepseek-v4p1-flash'
async def chat_call(cli, **body): return await provider_req(cli, '/chat/completions', body, stream=body.get('stream', False))

In [ ]:
kimi_cli = AsyncHttpCli('https://api.moonshot.ai/v1', base_headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})


In [ ]:
qwen_cli = AsyncHttpCli('https://api.fireworks.ai/inference/v1', base_headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})


In [ ]:
ds_cli = AsyncHttpCli('https://api.deepseek.com/v1', base_headers={"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}"})


In [ ]:
# qwen_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['QWEN_API_KEY']}"})
# for op in qwen_cli.ops: op.base_url = 'https://dashscope.aliyuncs.com/compatible-mode/v1'

## Normalize

These helpers read the first choice in a Chat Completions response. They convert its content, tool calls, finish reason, and usage to fastllm's shared types.

### Tool Calls

`norm_tool_calls` reads `message.tool_calls` for a completed response or `delta.tool_calls` for a stream. It parses complete JSON arguments into dicts. Streaming argument fragments instead use the `_delta` entry for the collector.

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add)
sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
#| export
def norm_tool_calls(resp, delta=False):
    "Extract Chat Completions tool calls as normalized tool calls, optionally from streaming delta events"
    out = []
    key = 'delta' if delta else 'message'
    if not   (tcs:= nested_idx(resp, 'choices', 0, key, 'tool_calls')): return out
    for tc in tcs:
        if not (fn:=tc.get("function")): continue
        raw  = {k:v for k,v in tc.items() if k not in ("id","function")}
        args = json.loads(fn.get("arguments")) if not delta else {'_delta': fn.get("arguments")}
        out.append(ToolUse(id=tc.get("id",""), name=fn.get("name",""), arguments=args, raw=raw))
    return out

User defined:

In [ ]:
resp = await chat_call(fw_cli, model=ds, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[sch])
norm_tool_calls(resp)

[ToolUse(raw={'type': 'function'}, cache_control=None, id='call_goAPO9vrPy8pa7hIjPfcEulD', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, text=None),
 ToolUse(raw={'type': 'function'}, cache_control=None, id='call_VmREQYlJx5KVeJypPTU51HUs', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, text=None)]

### Usage

This module imports `norm_usage` from `openai_responses`. The shared function accepts Chat's prompt/completion token names and cached/reasoning token details.

Text response:

In [ ]:
resp = await chat_call(fw_cli, model=ds, messages=[{"role": "user", "content": "Hi!"}], tools=[sch])
norm_usage(resp)

Usage(prompt_tokens=46, completion_tokens=10, total_tokens=56, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 46, 'completion_tokens': 10, 'total_tokens': 56, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

Tool call response:

In [ ]:
resp = await chat_call(fw_cli, model=ds, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[sch])
norm_usage(resp)

Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

### Finish Reason

`norm_finish` reads the first choice's `finish_reason`. A `stop` with pending client tool calls becomes `tool_calls`. Other values pass through.

In [ ]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    r = nested_idx(resp, 'choices', 0, 'finish_reason')
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

### Non-stream Completion

`norm_parts` reads `choices[0].message`. Nonempty `reasoning_content`, `content`, and `refusal` become `Thinking`, `Text`, and `Refusal` parts. Text annotations become `citations`. Tool calls follow those parts.

A missing message becomes an empty dict. Missing or empty text fields don't create parts. This is the Chat format, not the Responses API's `output` items or reasoning `summary` arrays.


In [ ]:
#| export
def norm_parts(resp):
    "Normalize chat.completions response object into Completion."
    msg = nested_idx(resp, 'choices', 0, 'message') or {}
    parts = []
    if thinking := msg.get('reasoning_content'): parts.append(Thinking(thinking))
    if cts := msg.get('content'): parts.append(Text(cts, citations=msg.get('annotations',[])))
    if ref := msg.get('refusal'): parts.append(Refusal(ref))
    return parts + norm_tool_calls(resp)

`raw_msg` retains the assistant message for replay, including `tool_calls` and any provider-specific `reasoning_content`. It returns that dict without copying or changing it.

In [ ]:
#| export
def raw_msg(resp):
    "The assistant message as the API sent it"
    return nested_idx(resp, 'choices', 0, 'message')

Register these normalizers under `openai_chat`. `mk_completion` uses them to construct the shared `Completion`, with `raw_msg` supplying its assistant message's replay data.

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, raw_msg=raw_msg)
api_registry.register('openai_chat', **norm_funcs)

#### Text

In [ ]:
mn,api_name,vnd_nm = ds, 'openai_chat', 'fireworks_ai'
resp = await chat_call(fw_cli, model=mn, messages=[{"role": "user", "content": "Say hi"}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hi there! How can I assist you today?

::: details

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

#### Tool call

In [ ]:
resp = await chat_call(fw_cli, model=mn, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],tools=[sch])
comp = mk_completion(resp,mn,api_name,vnd_nm)
comp



🔧 simple_add({'a': 3, 'b': 5})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `gpt-4o-mini`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

#### Refusal

The `refusal` field is separate from ordinary message text. This small input to `mk_completion` shows how it becomes a `Refusal` part.

In [ ]:
comp = mk_completion({"model": ds, "choices": [dict(index=0, message=dict(role="assistant", content=None, refusal="I can't help with that."), finish_reason="stop")], "usage": dict(prompt_tokens=10, completion_tokens=5, total_tokens=15)}, mn, api_name, vnd_nm)
comp

I can't help with that.

::: details

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=5, total_tokens=15, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 10, 'completion_tokens': 5, 'total_tokens': 15})`

:::

### Streaming

`norm_sse_event` reads text, reasoning, refusal, tool-call, finish, and usage fields into a `Delta`. `PartAccum` collects the deltas while the shared collector yields incremental parts. The final value is a `Completion`. Error events raise a transport API error.

In [ ]:
print_stream = PrintStream()

Chat streams can split tool arguments across chunks. Preserve each JSON fragment under `_delta` until the collector can parse the complete arguments:

```py
ToolUse(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, raw=raw)
```

A final usage event can have an empty `choices` list. The normalizer still reads its usage.

In [ ]:
#| export
def norm_sse_event(ev, **kwargs):
    "Normalize a chat completion stream event."
    # usage always arrives as a single final event with choices: []
    ev = obj2dict(ev)
    fin = nested_idx(ev, 'choices', 0, 'finish_reason')
    tcs = norm_tool_calls(ev, delta=True)
    if (dlt:=nested_idx(ev, 'choices', 0, 'delta')) is not None:
        text, thinking, refusal = dlt.get('content'), dlt.get('reasoning_content'), dlt.get('refusal')
    else: text, thinking, refusal = None,None,None
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=text, thinking=thinking, refusal=refusal, tool_calls=tcs, finish_reason=fin, usage=norm_usage(ev), raw=ev, **kwargs)

In [ ]:
ev = {'id': 'ea380758-a487-47fe-a746-b14dbb858cd5', 'object': 'chat.completion.chunk', 'created': 1777546979, 'model': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'choices': [dict(index=0, delta={'content': '', 'reasoning_content': None}, logprobs=None, finish_reason='stop')], 'usage': {'prompt_tokens': 35788, 'completion_tokens': 57, 'total_tokens': 35845, 'prompt_tokens_details': {'cached_tokens': 24448}, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_cache_hit_tokens': 24448, 'prompt_cache_miss_tokens': 11340}}

In [ ]:
norm_sse_event(ev)

Delta(text='', thinking=None, refusal=None, tool_calls=[], citations=[], finish_reason='stop', usage=Usage(prompt_tokens=35788, completion_tokens=57, total_tokens=35845, cached_tokens=24448, cache_creation_tokens=0, reasoning_tokens=32, search_queries=0, raw={'prompt_tokens': 35788, 'completion_tokens': 57, 'total_tokens': 35845, 'prompt_tokens_details': {'cached_tokens': 24448}, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_cache_hit_tokens': 24448, 'prompt_cache_miss_tokens': 11340}), raw={'id': 'ea380758-a487-47fe-a746-b14dbb858cd5', 'object': 'chat.completion.chunk', 'created': 1777546979, 'model': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'choices': [{'index': 0, 'delta': {'content': '', 'reasoning_content': None}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 35788, 'completion_tokens': 57, 'total_tokens': 35845, 'prompt_tokens_details': {'cached_tokens': 24448}, 'completion_tokens_details'

In [ ]:
ev = dict(id='chatcmpl-chatcmpl-2018e36c87e44315b017d8d8e320589f', object='chat.completion.chunk', created=1777548179,
    model='accounts/fireworks/models/qwen3p6-plus', choices=[], usage=dict(prompt_tokens=36355, total_tokens=36701, completion_tokens=346))

In [ ]:
norm_sse_event(ev)

Delta(text=None, thinking=None, refusal=None, tool_calls=[], citations=[], finish_reason=None, usage=Usage(prompt_tokens=36355, completion_tokens=346, total_tokens=36701, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 36355, 'total_tokens': 36701, 'completion_tokens': 346}), raw={'id': 'chatcmpl-chatcmpl-2018e36c87e44315b017d8d8e320589f', 'object': 'chat.completion.chunk', 'created': 1777548179, 'model': 'accounts/fireworks/models/qwen3p6-plus', 'choices': [], 'usage': {'prompt_tokens': 36355, 'total_tokens': 36701, 'completion_tokens': 346}})

`delta_index_fn` tells the collector which part to update. Tool calls use `tool_<index>` keys based on the provider's tool index.

For non-tool parts, the first index is zero. Consecutive deltas of the same kind share an index. A change of kind starts the next index. The function returns the current index and the index to retain for the next event.

In [ ]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if d.tool_calls: 
        tc_idx = nested_idx(d.tool_calls, 0, 'raw', 'index')
        return f"tool_{tc_idx}", last_idx
    if not (last_typ or last_idx): return 0,0
    if typ == last_typ: return last_idx, last_idx
    return last_idx + 1, last_idx + 1

`collate_raw` rebuilds an assistant message from stream chunks. It joins content, reasoning, and refusal text. It groups tool calls by index, retains their IDs, and joins name and argument fragments. The resulting tool-call list follows index order.

In [ ]:
#| export
def collate_raw(deltas):
    "The assistant message rebuilt from a stream's chunks"
    msg, tcs = dict(role='assistant', content=None), {}
    for ev in L(deltas).attrgot('raw'):
        d = nested_idx(ev, 'choices', 0, 'delta') or {}
        for k in ('content', 'reasoning_content', 'refusal'):
            if d.get(k): msg[k] = (msg.get(k) or '') + d[k]
        for tc in d.get('tool_calls') or []:
            t = tcs.setdefault(tc['index'], dict(id=tc.get('id'), type='function', function=dict(name='', arguments='')))
            if tc.get('id'): t['id'] = tc['id']
            fn = tc.get('function') or {}
            t['function']['name'] += fn.get('name') or ''
            t['function']['arguments'] += fn.get('arguments') or ''
    if tcs: msg['tool_calls'] = [tcs[i] for i in sorted(tcs)]
    return msg

`acollect_stream` combines this event normalizer with the shared collector and registers the result for `openai_chat`.

In [ ]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='openai_chat', **kwargs)
    async for o in res: yield o

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    acollect_stream=acollect_stream, raw_msg=raw_msg, collate_raw=collate_raw)
api_registry.register('openai_chat', **norm_funcs)

In [ ]:
api_registry['openai_chat']

namespace(finalize_usage=<function fastcore.imports.noop(x=None, *args, **kwargs)>,
          raw_msg=<function __main__.raw_msg(resp)>,
          collate_raw=<function __main__.collate_raw(deltas)>,
          norm_tool_calls=<function __main__.norm_tool_calls(resp, delta=False)>,
          norm_parts=<function __main__.norm_parts(resp)>,
          norm_finish=<function __main__.norm_finish(resp, tcs=None)>,
          norm_usage=<function fastllm.openai_responses.norm_usage(resp)>,
          acollect_stream=<function __main__.acollect_stream(resp, *, model=None, vendor_name=None, stop_callables=None)>)

Let's compare streamed and non-streamed completions.

#### Text

In [ ]:
mn,msgs = 'deepseek-flash',[{"role": "user", "content": "Say hi"}, 
    {"role": "assistant", "content": "Hello!"},
    {"role": "user", "content": "What did I just say?"}]
resp = await chat_call(ds_cli, model=mn,messages=msgs)
comp = mk_completion(resp, mn, 'openai_chat', 'deepseek')
comp


::: details
## Thinking

We need answer. User asks "What did I just say?" We need respond. They said "Say hi" then assistant said Hello. Then user asks "What did I just say?" Need answer accurately: You just said, "What did I just say?" Or maybe referring to previous message? The phrase "What did I just say?" is what they just said. Need maybe mention prior: Before that, you said "Say hi". But "just say" likely current message. We can answer: You just asked, “What did I just say?”. Keep concise.

:::

You just asked, “What did I just say?”

::: details

- model: `deepseek-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=44, completion_tokens=129, total_tokens=173, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=117, search_queries=0, raw={'prompt_tokens': 44, 'completion_tokens': 129, 'total_tokens': 173, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 117}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 44})`

:::

In [ ]:
mn,msgs = 'accounts/fireworks/models/deepseek-v4p1-flash',[{"role": "user", "content": "Say hi"}, 
    {"role": "assistant", "content": "Hello!"},
    {"role": "user", "content": "What did I just say?"}]
resp = await chat_call(qwen_cli, model=mn,messages=msgs)
comp = mk_completion(resp, mn, 'openai_chat', 'fireworks_ai')
comp


::: details
## Thinking

We need answer. User asks "What did I just say?" We need respond. We have conversation: user: "Say hi"; assistant: "Hello!" Then user: "What did I just say?" So they just said "What did I just say?" Also previously said "Say hi". Need answer likely "You asked: 'What did I just say?'" Maybe mention before that you said "Say hi". Need be accurate. They ask "What did I just say?" Current message is that. But perhaps they mean the previous message? In chat, "just say" means immediately prior to this? They just said "What did I just say?" Let's respond: "You just said, 'What did I just say?' Before that, you said, 'Say hi.'" That is good. Ensure concise.

:::

You just said: “What did I just say?”  
Before that, you said: “Say hi.”

::: details

- model: `accounts/fireworks/models/deepseek-v4p1-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=44, completion_tokens=192, total_tokens=236, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=168, search_queries=0, raw={'prompt_tokens': 44, 'total_tokens': 236, 'completion_tokens': 192, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 168}, 'output_tokens_details': {'reasoning_tokens': 168}})`

:::

#### Text

In [ ]:
mn,msgs = ds,[{"role": "user", "content": "Say hi"}]
resp = await chat_call(fw_cli, model=mn,messages=msgs)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hi there! How can I assist you today?

::: details

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

In [ ]:
resp = await chat_call(fw_cli, model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

Hi there! How can I assist you today?

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

#### Thinking

Reasoning-token counts don't imply that a provider exposes its reasoning text. This normalizer creates `Thinking` parts when a compatible endpoint supplies `reasoning_content`.

The original Kimi experiment used Fireworks's `accounts/fireworks/models/kimi-k2p5`. Its notes described thinking as enabled by default or disabled, without effort levels or token budgets. The commented experiment below now targets Moonshot's `kimi-k2.5` and remains disabled after a read timeout. Those calls don't establish its current reasoning-option support. Check the provider's model settings before enabling them.

In [ ]:
# mn,msgs = 'kimi-k2.5',[{"role": "user", "content": 'What is 31231231 * 12312?'}]
# resp = await chat_call(kimi_cli, model=mn,messages=msgs,reasoning_effort='low')
# comp = mk_completion(resp, mn, api_name, 'moonshot')

In [ ]:
# TODO: ReadTimeoutError
# resp = await chat_call(kimi_cli, model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
# async for o in acollect_stream(resp, vendor_name='moonshot'): print_stream(o)

In [ ]:
# test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
# test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
# test_eq(comp.finish_reason, o.finish_reason)
# test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name)
        test_eq(tc1.arguments,tc2.arguments)
        test_eq(tc1.server,tc2.server)

In [ ]:
mn = ds
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
resp = await chat_call(fw_cli, model=mn,messages=msgs, tools=[sch])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'a': 3, 'b': 5})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `gpt-4o-mini`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

In [ ]:
resp = await chat_call(fw_cli, model=mn,messages=msgs, tools=[sch], stream=True, stream_options={"include_usage": True})
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)


- ⏳ `simple_add(a="3", b="5")` ⏳

- ⏳ `simple_add(a="10", b="20")` ⏳


In [ ]:
test_tcs(comp.tool_calls,o.tool_calls)

In [ ]:
test_eq(len(comp.tool_calls), len(o.tool_calls))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

## Denormalize

The request converters turn shared messages and options into Chat Completions fields.

### Msgs

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Text(txt)])

User and assistant messages become Chat messages. A shared `tool` message can contain several results; each becomes a separate Chat tool message.

This first tool-result converter demonstrates text output. The exported version later adds an explicit check that rejects media lists.

In [ ]:
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    return dict(role='tool', tool_call_id=p.id, content=tool_text(p.text))

This first user-message converter handles text. The exported version later adds media inputs.

In [ ]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    if len(m.content) == 1 and isinstance(m.content[0], Text): return dict(role='user', content=m.content[0].text or '')
    return dict(role='user', content=[dict(type='text', text=p.text or '') for p in m.content if isinstance(p, Text)])

In [ ]:
#| export
def denorm_tool_use(p:ToolUse):
    "Convert canonical tool_use Part to OpenAI Chat tool_call dict."
    return dict(id=p.id, type='function', function=dict(name=p.name, arguments=json.dumps(p.arguments)))

def denorm_assistant(m:Msg):
    "The message as the API sent it, else its text and tool calls"
    if m.raw: return m.raw
    msg, texts = dict(role='assistant'), [p for p in m.content if isinstance(p, Text)]
    if texts: msg['content'] = texts[0].text if len(texts)==1 else [dict(type='text', text=p.text or '') for p in texts]
    if tcs := [denorm_tool_use(p) for p in m.content if isinstance(p, ToolUse)]: msg['tool_calls'] = tcs
    return msg

def denorm_tool(m:Msg):
    "Convert canonical tool Msg to list of OpenAI Chat tool messages."
    return [denorm_tool_result(p) for p in m.content if isinstance(p, ToolResult)]

def denorm_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to OpenAI Chat messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_user(m))
        elif m.role == 'assistant': res.append(denorm_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_tool(m))
    return res

`denorm_assistant` returns an existing `Msg.raw` unchanged. This retains provider-specific fields during a tool loop. Without `raw`, it constructs a message from `Text` and `ToolUse` parts. It doesn't reconstruct reasoning content from `Thinking` parts.

In [ ]:
wire = denorm_msgs([mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'), comp.message])
test_is(wire[1], comp.message.raw)
wire[1]

{'role': 'assistant',
 'content': None,
 'tool_calls': [{'id': 'call_goAPO9vrPy8pa7hIjPfcEulD',
   'type': 'function',
   'function': {'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}},
  {'id': 'call_VmREQYlJx5KVeJypPTU51HUs',
   'type': 'function',
   'function': {'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}}],
 'refusal': None,
 'annotations': []}

In [ ]:
msg1 = mk_user_msg('Hi!')
resp = await chat_call(fw_cli, model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

Hello! How can I assist you today?

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

### Tool Schemas

In [ ]:
#| export
def denorm_tool_schs(tools):
    "Convert tools to OpenAI Chat format; already-nested dicts pass through unchanged."
    out = []
    for t in tools:
        if isinstance(t, dict) and isinstance(t.get('function'), dict): out.append(t); continue
        fn = fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        f = dict(name=name, description=desc, parameters=params)
        if isinstance(t, dict) and 'strict' in t: f['strict'] = t['strict']
        out.append({'type':'function', 'function':f})
    return out

`denorm_tool_schs` converts a Responses-style schema into Chat's nested `function` dict. It retains `strict`. An already-nested dict passes through unchanged.

In [ ]:
flat = dict(type='function', name='py', description='run code', parameters={'type':'object'}, strict=True)
nested = {'type':'function', 'function':dict(name='py', description='run code', parameters={'type':'object'}, strict=True)}
test_eq(denorm_tool_schs([flat]), [nested])
test_eq(denorm_tool_schs([nested]), [nested])

### Tool Choice

In [ ]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to OpenAI Chat format."
    _tc_modes = {'auto', 'required', 'any', 'force', 'none', 'off', 'disabled'}
    if v is None: return None
    if v in _tc_modes: return v if v in ('auto','none','required') else dict(any='required', force='required', off='none', disabled='none')[v]
    return {'type': 'function', 'function': {'name': v}}

Modes

In [ ]:
test_eq(denorm_tool_choice('required'), 'required')

Tool name

In [ ]:
test_eq(denorm_tool_choice('simple_add'), {'type': 'function', 'function': {'name': 'simple_add'}})

None

In [ ]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

`denorm_reasoning` passes the value through as `reasoning_effort`. It doesn't translate effort names, token budgets, or a special `disable` option. Anthropic's adaptive-thinking converter is separate.

In [ ]:
#| export
def denorm_reasoning(v): 
    if v is None: return None
    return v  # passthrough as reasoning_effort param

### Web Search Options

In [ ]:
#| export
def denorm_web_search(v): return v

### System

In [ ]:
#| export
def denorm_system(sp, msgs): 
    msgs.insert(0, dict(role='system', content=sys_text(part_txt(sp))))
    return msgs

In [ ]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

`denorm_system` inserts the system prompt at the start of the message list. It modifies that list in place.

In [ ]:
msgs = denorm_msgs([msg1])
msgs = denorm_system(sp, msgs)
msgs

[{'role': 'system',
  'content': 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'},
 {'role': 'user', 'content': 'What are you?'}]

In [ ]:
resp = await chat_call(fw_cli, model=ds,messages=msgs,stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

I be a swashbucklin' AI, ready to assist ye on yer treasure hunt fer knowledge!

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=32, completion_tokens=21, total_tokens=53, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 32, 'completion_tokens': 21, 'total_tokens': 53, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

### Media Inputs

Media parts store a URL or data URL in `text` and a media type in `mime`. The Chat user-message converter accepts images, audio, and files. It rejects `InputVideo`.

A message containing only one `Text` part uses a plain `content` string. Mixed content uses a list of typed blocks.

In [ ]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    parts = []
    for p in m.content:
        if   isinstance(p, Text):       parts.append({"type": "text", "text": p.text or ""})
        elif isinstance(p, InputImage): parts.append(denorm_image(p))
        elif isinstance(p, InputAudio): parts.append(denorm_audio(p))
        elif isinstance(p, InputVideo): raise ValueError("OpenAI Chat API does not support video input")
        elif isinstance(p, InputFile):  parts.append(denorm_file(p))
    if len(parts) == 1 and parts[0].get('type') == 'text': return dict(role='user', content=parts[0]['text'])
    return dict(role='user', content=parts)

#### Image

In [ ]:
#| export
def denorm_image(p): return {"type": "image_url", "image_url": {"url": p.text}}

Image URL

In [ ]:
msg1 = Msg(role='user', content=[InputImage(sample_img_url), Text('What is this image?')])

In [ ]:
resp = await chat_call(fw_cli, model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

This image depicts a tranquil natural landscape featuring a serene lake surrounded by lush greenery and mountains. The scene includes a wooden deck in the foreground and reflects a peaceful atmosphere with mountains in the background. The overall composition captures the beauty of nature and is likely aimed at evoking feelings of calmness and serenity.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=25513, completion_tokens=60, total_tokens=25573, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 25513, 'completion_tokens': 60, 'total_tokens': 25573, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

Image data

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[InputImage(img_b64), Text('What color is this pixel?')])

In [ ]:
resp = await chat_call(fw_cli, model=mn,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

The color of the pixel appears to be a solid red.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=8513, completion_tokens=12, total_tokens=8525, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 8513, 'completion_tokens': 12, 'total_tokens': 8525, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

#### Audio

In [ ]:
#| export
def denorm_audio(p):
    _mime_audio_fmt = {'audio/wav':'wav', 'audio/mpeg':'mp3', 'audio/mp3':'mp3'}
    if not (b64:=data_url(p.text)): raise ValueError("OpenAI Chat audio input requires base64 data URL")
    return {"type": "input_audio", "input_audio": {"data": b64[1], "format": _mime_audio_fmt.get(b64[0], 'wav')}}

In [ ]:
wav_data = httpx2.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

Audio input needs an [audio-capable Chat model](https://developers.openai.com/api/docs/guides/audio?example=audio-in#add-audio-to-your-existing-application). This example uses `gpt-audio-1.5`; older examples used `gpt-4o-audio-preview`. The converter requires a base64 data URL and maps WAV/MP3 MIME types to the API's format names.

In [ ]:
msg1 = Msg(role='user', content=[InputAudio(audio_b64), Text('What is this audio saying?')])

In [ ]:
resp = await chat_call(oai_cli, model='gpt-audio-1.5',messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name='openai_chat'): pass
comp

{"transcription": "The sun rises in the east and sets in the west. This simple fact has been observed by humans for thousands of years."}

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=86, completion_tokens=31, total_tokens=117, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 86, 'completion_tokens': 31, 'total_tokens': 117, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 69, 'text_tokens': 17, 'image_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': 31}})`

:::

#### Files

`denorm_file` requires a base64 data URL. It derives a filename such as `upload.pdf` from the MIME type. Ordinary file URLs raise `ValueError`. Anthropic and Gemini instead include the MIME type in their base64 file blocks.

In [ ]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"type": "file", "file": {"file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}}
    raise ValueError("OpenAI Chat file input requires base64 data URL or file_id, not URLs")

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[InputFile(pdf_url), Text('What is the title of this paper?')])

In [ ]:
try: await chat_call(oai_cli, model=luna,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
except Exception as e: print(e)

OpenAI Chat file input requires base64 data URL or file_id, not URLs


In [ ]:
pdf_b64 = f"data:application/pdf;base64,{base64.b64encode(httpx2.get(pdf_url).content).decode()}"
msg1 = Msg(role='user', content=[InputFile(pdf_b64), Text('What is the title of this paper')])

In [ ]:
resp = await chat_call(oai_cli, model=luna,messages=denorm_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for comp in acollect_stream(resp, vendor_name='openai_chat'): pass
comp

The title of the paper is "Attention Is All You Need."

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=11386, completion_tokens=13, total_tokens=11399, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'prompt_tokens': 11386, 'completion_tokens': 13, 'total_tokens': 11399, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

:::

### Media Tool Call Results

This Chat converter rejects any list-valued tool result, including images or files. Other providers have structured media-result formats. Non-list results use `tool_text`.

The example first obtains a real tool call, then checks that the converter rejects its image result.


In [ ]:
#| export
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    if isinstance(p.text, list): raise ValueError("OpenAI Chat does not support media in tool results")
    return dict(role='tool', tool_call_id=p.id or (p.raw or {}).get('call_id', ''), content=tool_text(p.text))

In [ ]:
def test_img(): "Show the test image"
img_tools = denorm_tool_schs([lite_mk_func(test_img)])
img_q = mk_user_msg('Call test_img, then describe what it shows in one sentence.')
resp = await chat_call(fw_cli, model=mn, messages=denorm_msgs([img_q]), tools=img_tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)
msgs = [img_q, comp.message, Msg(role='tool', content=[ToolResult(id=comp.tool_calls[0].id, name='test_img', text=[InputImage(img_b64)])])]
comp.tool_calls


[ToolUse(raw={'type': 'function'}, cache_control=None, id='call_sVAfcqmuz5TXOr92brF0DRZs', name='test_img', arguments={}, server=False, text=None)]

In [ ]:
with expect_fail(ValueError, 'media'): denorm_msgs(msgs)


## Payload

`mk_payload` adds the converted messages and shared request options. Streaming requests include `stream_options={'include_usage': True}`. Reasoning and web-search options pass through in their Chat fields. Explicit `False` for parallel tool calls and zero temperature remain in the payload.

In [ ]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(model=model, messages=denorm_msgs(msgs))
    if sp:=kwargs.get('system'):            payload['messages'] = denorm_system(sp, payload['messages'])
    if kwargs.get('stream'):                payload.update(stream=True, stream_options={"include_usage": True})
    if mt:=kwargs.get('max_tokens'):        payload['max_tokens'] = mt
    if tools:=kwargs.get('tools'):          payload['tools'] = denorm_tool_schs(tools)
    if tchc:=kwargs.get('tool_choice'):     payload['tool_choice'] = denorm_tool_choice(tchc)
    if (ptc:=kwargs.get('parallel_tool_calls')) is not None: payload['parallel_tool_calls'] = ptc
    if thk:=kwargs.get('reasoning_effort'): payload['reasoning_effort'] = denorm_reasoning(thk)
    if (wopts:=kwargs.get('web_search_options')) is not None:  payload['web_search_options'] = denorm_web_search(wopts)
    if (temp:=kwargs.get('temperature')) is not None: payload['temperature'] = temp
    return payload

`get_hdrs` prefers an OAuth bearer token. Otherwise it uses the explicit API key or `OPENAI_API_KEY`.

In [ ]:
#| export
def get_hdrs(api_key=None, oauth_token=None):
    "Bearer auth: an OAuth access token, or else an API key from the argument or the environment"
    return {"Authorization": f"Bearer {oauth_token or get_api_key(api_key, 'OPENAI_API_KEY')}"}


## Cost

`cost` separates ordinary text tokens, cached input tokens, and input/output audio tokens. It multiplies each count by the corresponding model rate. Missing cache or audio rates default to zero.

In [ ]:
#| export
def cost(usage, m):
    raw = usage.raw
    pd,cd = raw.get('prompt_tokens_details') or {},raw.get('completion_tokens_details') or {}
    cached = pd.get('cached_tokens', 0)
    in_audio, out_audio = pd.get('audio_tokens', 0), cd.get('audio_tokens', 0)
    in_txt  = raw['prompt_tokens']     - cached - in_audio
    out_txt = raw['completion_tokens'] - out_audio
    cost  = in_txt  * m.input_cost_per_token  + out_txt * m.output_cost_per_token
    cost += cached  * m.get('cache_read_input_token_cost', 0)
    cost += in_audio  * m.get('input_cost_per_audio_token', 0)
    cost += out_audio * m.get('output_cost_per_audio_token', 0)
    return cost

## Register API

The `openai_chat` registry entry supplies the normalizers, request builder, stream collector, replay helpers, authentication headers, and cost function. `acomplete` uses `/chat/completions` as the endpoint.

`fix_payload` runs after payload construction. If the final message is from the assistant, DeepSeek's `deepseek-flash` model gets `prefix=True`. Moonshot models whose name contains `kimi` get `partial=True`. These flags mark an assistant prefill.

In [ ]:
#| export
def fix_payload(payload, model, vendor_name):
    "Apply vendor request quirks to a built payload."
    if nested_idx(payload, 'messages', -1, 'role') == 'assistant':
        if vendor_name == 'deepseek' and model == 'deepseek-flash':   payload['messages'][-1]['prefix'] = True
        if vendor_name == 'moonshot' and 'kimi' in model: payload['messages'][-1]['partial'] = True

In [ ]:
p = dict(messages=[dict(role='user', content='q'), dict(role='assistant', content='a')])
fix_payload(p, 'deepseek-flash', 'deepseek')
test_eq(p['messages'][-1]['prefix'], True)
fix_payload(p, 'kimi-k3', 'moonshot')
test_eq(p['messages'][-1]['partial'], True)
p2 = dict(messages=[dict(role='user', content='q')])
fix_payload(p2, 'deepseek-flash', 'deepseek')
test_eq(p2, dict(messages=[dict(role='user', content='q')]))

In [ ]:
#| export
api_ns = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    raw_msg=raw_msg, collate_raw=collate_raw, acollect_stream=acollect_stream, mk_payload=mk_payload, cost=cost, get_hdrs=get_hdrs, fix_payload=fix_payload,
    endpoint='/chat/completions')
api_registry.register('openai_chat', **api_ns)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()